# Google - The Gemma 4 Developer Agent Competition
## Notebook 03: Submission Packager & Verifier Launcher

This launcher notebook packages your solution into a competition-compliant `submission.zip`:
1. Scaffolds declarative `agent.yaml`, sub-agents, and prompts.
2. Validates ADK security constraints (No custom python entrypoints, closed `ToolRegistry`).
3. Verifies model consistency across all agent YAMLs.
4. Produces the final `submission.zip` archive ready for upload.

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline Import Bootstrap (Compatible with both Local and Kaggle Datasets)
# ---------------------------------------------------------------------------
import os
import sys
from pathlib import Path

# Detect environment and resolve pipeline location
LOCAL_ROOT = Path("..").resolve()
KAGGLE_INPUT = Path("/kaggle/input")

candidate_paths = [
    LOCAL_ROOT,
    LOCAL_ROOT / "pipeline",
    Path("."),
    Path("pipeline")
]

# Check Kaggle dataset mounts
if KAGGLE_INPUT.exists():
    for dataset_dir in KAGGLE_INPUT.glob("*"):
        if (dataset_dir / "pipeline").exists():
            candidate_paths.insert(0, dataset_dir)
        elif (dataset_dir / "main.py").exists() and "pipeline" in dataset_dir.name.lower():
            candidate_paths.insert(0, dataset_dir.parent)

pipeline_loaded = False
for p in candidate_paths:
    if (p / "pipeline" / "main.py").exists() or (p / "pipeline").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print(f"Successfully loaded pipeline from: {p}")
        pipeline_loaded = True
        break

if not pipeline_loaded:
    print("[WARN] Pipeline folder not auto-detected; defaulting to current sys.path.")

from pipeline.main import PipelineRunner, status, train, evaluate, package
runner = PipelineRunner()
runner.status()

### 1. Build and Validate Submission Archive
One-line compilation via `runner.package()`.

In [ ]:
# Compile submission package
# Optional: specify adapter_path if submitting fine-tuned LoRA weights
submission_path = runner.package(
    model_name="gemma-4-12b-it",
    output_zip="../submission.zip",
    adapter_path=None
)

print(f"Packaged Archive: {submission_path}")
print(f"Archive Size: {submission_path.stat().st_size / 1024:.2f} KB")

### 2. Inspect Submission Contents
Verify root `agent.yaml` is present and no prohibited scripts exist.

In [ ]:
import zipfile

with zipfile.ZipFile(submission_path, "r") as zf:
    print("Files inside submission.zip:")
    for name in sorted(zf.namelist()):
        print(f"  - {name}")